In [ ]:
import pandas as pd
import numpy as np

# 1. Đọc dữ liệu từ file gốc
file_path = '/content/shipments_realistic.csv'
df = pd.read_csv(file_path)

print("Kích thước dữ liệu gốc:", df.shape)
print("\nSố lượng giá trị thiếu ở mỗi cột ban đầu:")
print(df.isnull().sum())

# Kiểm tra số lượng order_id bị trùng lặp (Sửa 'false' thành 'False')
duplicate_order_ids = df[df.duplicated(subset=['order_id'], keep=False)]['order_id'].nunique()
print(f"\nSố lượng order_id bị trùng lặp: {duplicate_order_ids}")

In [ ]:
# 2. Xử lý trùng lặp: Giữ lại dòng có đầy đủ dữ liệu nhất cho mỗi order_id
# Tính số lượng giá trị không bị khuyết (non-null) cho mỗi hàng
df['non_null_count'] = df.notnull().sum(axis=1)

# Sắp xếp giảm dần theo 'non_null_count' để dòng đầy đủ dữ liệu nhất lên đầu
df_sorted = df.sort_values(by='non_null_count', ascending=False)

# Loại bỏ trùng lặp theo 'order_id', giữ lại dòng đầu tiên (chính là dòng nhiều thông tin nhất)
df_cleaned = df_sorted.drop_duplicates(subset=['order_id'], keep='first').copy()

# Bỏ cột phụ trợ
df_cleaned = df_cleaned.drop(columns=['non_null_count'])

print("Kích thước sau khi loại bỏ trùng lặp order_id:", df_cleaned.shape)

In [ ]:
# 3. Điền khuyết dữ liệu thiếu (Imputation) theo các phương pháp phổ biến
print("Dữ liệu khuyết thiếu còn lại cần xử lý:")
missing_info = df_cleaned.isnull().sum()
print(missing_info[missing_info > 0])

# Lặp qua các cột để điền khuyết dựa trên kiểu dữ liệu
for col in df_cleaned.columns:
    if df_cleaned[col].isnull().any():
        # Nếu cột là kiểu số (float, int)
        if pd.api.types.is_numeric_dtype(df_cleaned[col]):
            median_value = df_cleaned[col].median()
            df_cleaned[col] = df_cleaned[col].fillna(median_value)
            print(f"- Điền khuyết cột số '{col}' bằng Median: {median_value}")
        # Nếu cột là kiểu phân loại / chuỗi
        else:
            mode_value = df_cleaned[col].mode()
            if not mode_value.empty:
                fill_val = mode_value[0]
                df_cleaned[col] = df_cleaned[col].fillna(fill_val)
                print(f"- Điền khuyết cột phân loại '{col}' bằng Mode: {fill_val}")
            else:
                df_cleaned[col] = df_cleaned[col].fillna("Unknown")
                print(f"- Điền khuyết cột phân loại '{col}' bằng 'Unknown'")

# Kiểm tra lại dữ liệu thiếu sau khi điền
assert df_cleaned.isnull().sum().sum() == 0, "Vẫn còn giá trị thiếu chưa xử lý!"

In [ ]:
# 4. Lọc các cột theo đúng schema yêu cầu của bảng SHIPMENT và xuất ra file mới
columns_to_keep = ['order_id', 'shipper_id', 'ship_date', 'delivery_date', 'shipping_fee']
shipment_df = df_cleaned[columns_to_keep].copy()

output_path = '/content/shipments_new.csv'
shipment_df.to_csv(output_path, index=False)

print(f"Đã xuất file thành công theo schema mới tại: {output_path}")
print("Kích thước dữ liệu file mới:", shipment_df.shape)
print("\nXem trước 5 dòng đầu tiên của file mới:")
display(shipment_df.head())